# R1-06 · Estadística descriptiva — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R1 · *Analista de Datos* · Semana 7 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Comparar **media y mediana** y leer la asimetría.
2. Medir **dispersión**: rango, desviación estándar y coeficiente de variación.
3. Usar **percentiles, cuartiles e IQR**, y detectar **atípicos** sin confundirlos con errores.
4. Distinguir **promedio de razones** y **razón de totales** (estudiantes por curso).
5. **Comparar grupos** (rural/urbano) sin dejarse engañar por un promedio.

**Competencia de salida:** resumir e interpretar una variable con varias medidas a la vez, y comunicar el número correcto.

**Dato real:** matrícula oficial 2025 por establecimiento (MINEDUC): estudiantes y cursos de 10.945 establecimientos.

**Entregable:** que las **8 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "matricula.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/matricula.csv", CSV)

df = pd.read_csv(CSV)
print("Datos cargados:", df.shape, "filas x columnas")
df.head()

## ¿Cuántos estudiantes tiene un establecimiento «típico»?

Parece una pregunta simple. Pero en Chile conviven escuelas rurales de **un solo estudiante** con
liceos de **casi 4.000**. Un único número —el promedio— puede contar una historia muy distinta a
la realidad de la mayoría. La **estadística descriptiva** resume una columna con varias medidas a
la vez:

- **Centro:** ¿alrededor de qué valor se agrupan? (media, mediana)
- **Dispersión:** ¿qué tan separados están? (rango, desviación estándar, coeficiente de variación)
- **Posición:** ¿cómo se reparten? (cuartiles, percentiles, rango intercuartílico)

Trabajaremos con `mat_total` (estudiantes por establecimiento) y `cursos` de la matrícula oficial
2025: **10.945 establecimientos**.

## 1. Centro: media y mediana

- **Media** (`.mean()`): suma todo y reparte en partes iguales. **Sensible a los extremos**:
  un liceo gigante la arrastra hacia arriba.
- **Mediana** (`.median()`): el valor del medio cuando ordenas. La mitad queda abajo, la mitad
  arriba. Es **robusta**: los extremos casi no la mueven.

Cuando hay **muchos valores chicos y pocos muy grandes** (asimetría a la derecha), la media queda
**por encima** de la mediana. Compararlas es la primera alarma.

### ✍️ Ejercicio 1 — Media y mediana de la matrícula

In [ ]:
media = df["mat_total"].mean()
mediana = df["mat_total"].median()
razon = media / mediana
print(f"media: {media:.1f} | mediana: {mediana:.0f} | media/mediana: {razon:.2f}")

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert abs(media - df["mat_total"].mean()) < 1e-6, "media = df['mat_total'].mean()"
    assert abs(mediana - df["mat_total"].median()) < 1e-6, "mediana = df['mat_total'].median()"
    assert abs(razon - media / mediana) < 1e-9
    assert razon > 1, "La media debería quedar sobre la mediana (asimetría a la derecha)"
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", '.mean(), .median() y razon = media / mediana.')
    print("   Detalle:", e)

## 2. Dispersión: rango, desviación estándar y coeficiente de variación

- **Rango** = máximo − mínimo: rápido, pero depende solo de los dos extremos.
- **Desviación estándar** (`.std()`): cuánto se alejan los datos de la media, **en las mismas
  unidades** (estudiantes).
- **Coeficiente de variación** (CV) = desviación / media: la dispersión **relativa**, sin
  unidades. Permite comparar variables distintas. Un CV mayor que 1 indica que la desviación
  supera a la propia media: datos **muy** heterogéneos.

### ✍️ Ejercicio 2 — Mide la dispersión

In [ ]:
rango = df["mat_total"].max() - df["mat_total"].min()
desviacion = df["mat_total"].std()
cv = desviacion / df["mat_total"].mean()
print(f"rango: {rango} | desviación: {desviacion:.1f} | CV: {cv:.2f}")

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    _s = df["mat_total"]
    assert rango == _s.max() - _s.min(), "rango = máximo − mínimo"
    assert abs(desviacion - _s.std()) < 1e-6, "desviacion = df['mat_total'].std()"
    assert abs(cv - _s.std() / _s.mean()) < 1e-9, "cv = desviación / media"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'df["mat_total"].max() - df["mat_total"].min(); .std(); cv = desviacion / media.')
    print("   Detalle:", e)

## 3. Posición: cuartiles, percentiles y rango intercuartílico

Los **percentiles** cortan los datos ordenados: el percentil 90 deja al 90% por debajo. Los
**cuartiles** son los percentiles 25 (Q1), 50 (la mediana) y 75 (Q3). Se calculan con
`.quantile(0.25)`, `.quantile(0.75)`, etc.

El **rango intercuartílico** (IQR = Q3 − Q1) mide el ancho del **50% central**: a diferencia del
rango, no le importan los extremos. `describe()` muestra casi todo de una vez:

In [ ]:
df["mat_total"].describe()

### ✍️ Ejercicio 3 — Cuartiles, IQR y percentil 90

In [ ]:
q1 = df["mat_total"].quantile(0.25)
q3 = df["mat_total"].quantile(0.75)
iqr = q3 - q1
p90 = df["mat_total"].quantile(0.90)
print(f"Q1: {q1:.0f} | Q3: {q3:.0f} | IQR: {iqr:.0f} | P90: {p90:.0f}")
print(f"La mitad central de los establecimientos tiene entre {q1:.0f} y {q3:.0f} estudiantes.")

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    _s = df["mat_total"]
    assert q1 == _s.quantile(0.25) and q3 == _s.quantile(0.75), "Usa .quantile(0.25) y .quantile(0.75)"
    assert iqr == q3 - q1, "iqr = q3 - q1"
    assert p90 == _s.quantile(0.90), "p90 = .quantile(0.90)"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", '.quantile(0.25), .quantile(0.75), iqr = q3 - q1, .quantile(0.90).')
    print("   Detalle:", e)

## 4. Atípicos con la regla del IQR (atípico ≠ error)

Una regla estándar marca como **atípico** todo valor sobre `Q3 + 1,5 × IQR` (o bajo
`Q1 − 1,5 × IQR`). Es la regla que dibujan los gráficos de caja.

Pero ojo: **atípico no significa error**. En R1-04 un -1 era imposible y había que corregirlo;
aquí un liceo con 3.000 estudiantes es raro, pero **real**. Lo que corresponde es mirarlo, no
borrarlo.

### ✍️ Ejercicio 4 — Cuenta los atípicos y mira el más grande

In [ ]:
limite_sup = q3 + 1.5 * iqr
atipicos = df[df["mat_total"] > limite_sup]
n_atipicos = len(atipicos)
el_mas_grande = df.loc[df["mat_total"].idxmax(), "nombre"]
print(f"límite: {limite_sup:.0f} estudiantes | atípicos: {n_atipicos} ({n_atipicos / len(df):.1%})")
print("el más grande:", el_mas_grande)
atipicos.sort_values("mat_total", ascending=False)[["nombre", "comuna", "dependencia", "mat_total"]].head()

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    _s = df["mat_total"]
    _q1, _q3 = _s.quantile(0.25), _s.quantile(0.75)
    _lim = _q3 + 1.5 * (_q3 - _q1)
    assert abs(limite_sup - _lim) < 1e-9, "limite_sup = q3 + 1.5 * iqr"
    assert n_atipicos == int((_s > _lim).sum()), "Cuenta las filas con mat_total > limite_sup"
    assert el_mas_grande == df.loc[_s.idxmax(), "nombre"], "Usa df.loc[df['mat_total'].idxmax(), 'nombre']"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'limite_sup = q3 + 1.5 * iqr ; atipicos = df[df["mat_total"] > limite_sup] ; idxmax() para el más grande.')
    print("   Detalle:", e)

## 5. Leer el sesgo: ¿cuántos están sobre el promedio?

Si el promedio fuera un buen «valor típico», más o menos la mitad de los establecimientos
estaría por encima. Con datos asimétricos, **la mayoría queda por debajo**: unos pocos grandes
levantan el promedio.

### ✍️ Ejercicio 5 — Proporción de establecimientos sobre el promedio

In [ ]:
pct_sobre_promedio = 100 * (df["mat_total"] > df["mat_total"].mean()).mean()
print(f"{pct_sobre_promedio:.1f}% de los establecimientos supera el promedio de {df['mat_total'].mean():.0f}")

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    _esp = 100 * (df["mat_total"] > df["mat_total"].mean()).mean()
    assert abs(pct_sobre_promedio - _esp) < 1e-6, "100 * (df['mat_total'] > df['mat_total'].mean()).mean()"
    assert pct_sobre_promedio < 50
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", '100 * (df["mat_total"] > df["mat_total"].mean()).mean()')
    print("   Detalle:", e)

## 6. Estudiantes por curso: promedio de razones vs razón de totales

`mat_total / cursos` aproxima el **tamaño de curso** de cada establecimiento. Hay dos formas de
resumirlo y **no dan lo mismo**:

- **Promedio de razones**: calculas la razón en cada establecimiento y promedias. Cada
  establecimiento pesa igual → responde «¿cómo es el curso en el **establecimiento** típico?».
- **Razón de totales**: total de estudiantes / total de cursos. Cada **estudiante** pesa igual →
  responde «¿en qué tamaño de curso está el **estudiante** típico?».

Como las escuelas pequeñas (rurales, multigrado) tienen cursos chicos y son muchas, el primer
número sale más bajo. Ninguno está «mal»: hay que **decir cuál usaste**.

### ✍️ Ejercicio 6 — Calcula ambas medidas

In [ ]:
df["est_por_curso"] = df["mat_total"] / df["cursos"]
promedio_de_razones = df["est_por_curso"].mean()
razon_de_totales = df["mat_total"].sum() / df["cursos"].sum()
print(f"promedio de razones: {promedio_de_razones:.1f} | razón de totales: {razon_de_totales:.1f} estudiantes por curso")

In [ ]:
# ── Celda de chequeo · Ejercicio 6 ──────────────────────────────────────────
try:
    _r = df["mat_total"] / df["cursos"]
    assert abs(df["est_por_curso"] - _r).max() < 1e-9, "est_por_curso = mat_total / cursos"
    assert abs(promedio_de_razones - _r.mean()) < 1e-9
    assert abs(razon_de_totales - df["mat_total"].sum() / df["cursos"].sum()) < 1e-9, "Suma todos los estudiantes y divide por la suma de todos los cursos"
    assert razon_de_totales > promedio_de_razones
    print("✅ Ejercicio 6: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'df["mat_total"] / df["cursos"]; luego .mean() de esa columna vs df["mat_total"].sum() / df["cursos"].sum().')
    print("   Detalle:", e)

## 7. Comparar grupos: rural vs urbano

Para comparar grupos, `groupby` + `agg` calcula varias medidas a la vez. Compara el tamaño de
curso en establecimientos **rurales** (`rural = 1`) y **urbanos** (`rural = 0`) mirando la
cantidad, la media **y** la mediana.

### ✍️ Ejercicio 7 — Tamaño de curso por ruralidad

In [ ]:
resumen = df.groupby("rural")["est_por_curso"].agg(["count", "mean", "median"]).round(1)
brecha = resumen.loc[0, "median"] / resumen.loc[1, "median"]
print(resumen)
print(f"El curso urbano típico es {brecha:.1f} veces el rural típico (medianas).")

In [ ]:
# ── Celda de chequeo · Ejercicio 7 ──────────────────────────────────────────
try:
    _g = df.groupby("rural")["est_por_curso"]
    assert list(resumen.columns) == ["count", "mean", "median"], "Usa .agg(['count', 'mean', 'median'])"
    assert abs(resumen.loc[1, "median"] - _g.median()[1]) < 0.06
    assert abs(brecha - _g.median()[0] / _g.median()[1]) < 0.02, "brecha = mediana urbana / mediana rural"
    print("✅ Ejercicio 7: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'df.groupby("rural")["est_por_curso"].agg(["count", "mean", "median"]); brecha con resumen.loc[0, "median"] / resumen.loc[1, "median"].')
    print("   Detalle:", e)

## 8. Interpretar sin engañarse: ¿en qué grupo miente más el promedio?

Usa la razón **media/mediana** de `mat_total` en cada grupo: mientras más lejos de 1, más
asimétrico el grupo y más engaña su promedio. Guarda en `razones` una Serie con la razón para
rural (1) y urbano (0), y elige la conclusión en `conclusion`:

- **A.** El grupo **urbano** es el más asimétrico: su promedio exagera el tamaño de la escuela urbana típica.
- **B.** El grupo **rural** es el más asimétrico: unas pocas escuelas rurales grandes inflan su
  promedio, y la escuela rural típica es mucho más pequeña de lo que el promedio sugiere.
- **C.** Ambos grupos son simétricos (razón cercana a 1): el promedio describe bien a los dos.

### ✍️ Ejercicio 8 — Razón media/mediana por grupo

In [ ]:
g = df.groupby("rural")["mat_total"]
razones = g.mean() / g.median()
print(pd.DataFrame({"media": g.mean().round(1), "mediana": g.median(), "media/mediana": razones.round(2)}))
conclusion = "B"

In [ ]:
# ── Celda de chequeo · Ejercicio 8 ──────────────────────────────────────────
try:
    _g = df.groupby("rural")["mat_total"]
    _raz = _g.mean() / _g.median()
    assert abs(razones[0] - _raz[0]) < 1e-9 and abs(razones[1] - _raz[1]) < 1e-9, "razones = g.mean() / g.median()"
    _letra = "C" if (abs(_raz - 1) < 0.15).all() else ("B" if _raz[1] > _raz[0] else "A")
    assert str(conclusion).strip().upper() == _letra, "Mira cuál de las dos razones está más lejos de 1"
    print("✅ Ejercicio 8: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'razones = g.mean() / g.median(); el grupo con la razón más alta es el más asimétrico.')
    print("   Detalle:", e)

## ✍️ Reflexión (no se corrige, pero es lo más importante)

Un informe dice: «La escuela rural chilena tiene en promedio 89 estudiantes». Con lo que viste hoy,
¿qué número le reportarías a una autoridad que decide el financiamiento de escuelas rurales, y cómo
lo acompañarías (mediana, percentiles, cantidad de escuelas) para que no se lleve una idea equivocada?

## Errores típicos

- **Reportar solo el promedio** con datos asimétricos → acompáñalo de la mediana.
- **Borrar atípicos porque «se ven raros»** → atípico ≠ error; mira primero si es real.
- **Promediar razones sin decirlo** → el promedio de razones y la razón de totales responden preguntas distintas.
- **Comparar grupos solo por la media** → compara también mediana y cantidad de casos.
- **Olvidar las unidades** → la desviación está en estudiantes; el CV no tiene unidades.

## Cierre

- Centro (media/mediana), dispersión (rango, desviación, CV) y posición (cuartiles, IQR, percentiles): **siempre juntos**.
- La razón media/mediana es una alarma rápida de asimetría.
- Comparar grupos sin engañarse = misma medida, la correcta, y declarada.

> *En R1-07 vas a ver estas mismas distribuciones en gráficos: histogramas y cajas que muestran
> de un vistazo lo que hoy calculaste.*